<a href="https://colab.research.google.com/github/Hajer5503/AgriSmart/blob/feature%2Fcrop-disease-detection/modules/crop_disease_detection/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Disease Detection – Data Preprocessing

This notebook prepares the dataset for model training.

Objectives:

- Build structured dataset with labels (crop + disease)
- Remove corrupted or invalid images
- Filter extremely small images
- Encode class labels
- Perform stratified train/validation/test split
- Define augmentation and preprocessing transforms
- Prepare PyTorch Dataset and DataLoader objects

This notebook ensures a clean and reproducible data pipeline before modeling.

In [1]:
import os
import random
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from google.colab import drive
from pathlib import Path

In [2]:
drive.mount('/content/drive')
DATASET_PATH = Path("/content/drive/MyDrive/Crop disease")
print(DATASET_PATH.exists())

Mounted at /content/drive
True


## 1. Build Structured Dataset Table

We generate a dataframe containing:

- Image path
- Combined label (crop + disease)

In [3]:
image_paths = list(DATASET_PATH.glob("**/*.jpg")) + \
              list(DATASET_PATH.glob("**/*.png")) + \
              list(DATASET_PATH.glob("**/*.jpeg"))

records = []

for path in image_paths:
    crop = path.parent.parent.name
    disease = path.parent.name
    label = f"{crop}_{disease}"
    records.append((path, label))

df = pd.DataFrame(records, columns=["path", "label"])

print("Total images found:", len(df))
df.head()

Total images found: 33641


,path,label
0,/content/drive/MyDrive/Crop disease/Wheat/Whea...,Wheat_Wheat___Yellow_Rust
1,/content/drive/MyDrive/Crop disease/Wheat/Whea...,Wheat_Wheat___Yellow_Rust
2,/content/drive/MyDrive/Crop disease/Wheat/Whea...,Wheat_Wheat___Yellow_Rust
3,/content/drive/MyDrive/Crop disease/Wheat/Whea...,Wheat_Wheat___Yellow_Rust
4,/content/drive/MyDrive/Crop disease/Wheat/Whea...,Wheat_Wheat___Yellow_Rust


## 2. Dataset Integrity Checks

We verify:

- Images are readable
- Images are not extremely small
- Corrupted files are excluded

Raw data remains untouched to preserve reproducibility.

In [4]:
clean_records = []
bad_images = []

for path, label in records:
    try:
        with Image.open(path) as img:
            w, h = img.size

            # Filter extremely small images
            if w >= 50 and h >= 50:
                clean_records.append((path, label))
            else:
                bad_images.append(path)

    except:
        bad_images.append(path)

df = pd.DataFrame(clean_records, columns=["path", "label"])

print("Clean images:", len(df))
print("Removed images:", len(bad_images))

Clean images: 33464
Removed images: 177


## 3. Label Encoding

Convert string labels to numerical indices for model training.

In [5]:
label_to_idx = {label: idx for idx, label in enumerate(sorted(df["label"].unique()))}
idx_to_label = {v: k for k, v in label_to_idx.items()}

df["label_idx"] = df["label"].map(label_to_idx)

n_classes = len(label_to_idx)

print("Number of classes:", n_classes)

Number of classes: 32


## 4. Stratified Splitting

We split the dataset:

- 70% training
- 15% validation
- 15% test

Stratification ensures class proportions are preserved.

In [6]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label_idx"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_idx"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 23424
Validation size: 5020
Test size: 5020


## 5. Class Imbalance Handling

If imbalance ratio > 5, class-weighted loss will be used during training.

In [7]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_idx"]),
    y=train_df["label_idx"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)

class_weights

tensor([ 0.6050,  2.6715,  0.7641,  0.7500,  0.6219,  0.3996,  0.7059,  1.0310,
         0.8663,  0.8683, 15.5745,  0.9161,  9.1500,  3.6784,  1.5346,  5.0833,
         1.1142,  1.0472,  0.8375,  1.0765,  1.7063,  0.7025,  1.0702,  1.0457,
         2.2317,  0.8079,  2.0333,  0.3812,  1.3382,  1.2730,  0.9266,  1.1712])

## 6. Data Transformations

All images will be resized to 224×224.

Training set includes augmentation.
Validation and test sets use deterministic preprocessing.

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

## 7. Custom Dataset Class

In [9]:
class CropDiseaseDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        label = row["label_idx"]

        if self.transform:
            img = self.transform(img)

        return img, label

## 8. DataLoader Preparation

In [10]:
train_dataset = CropDiseaseDataset(train_df, transform=train_transform)
val_dataset = CropDiseaseDataset(val_df, transform=val_transform)
test_dataset = CropDiseaseDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("DataLoaders ready.")

DataLoaders ready.


## 9. Summary

✔ Cleaned dataset  
✔ Removed corrupted / tiny images  
✔ Encoded labels  
✔ Stratified train/val/test split  
✔ Computed class weights  
✔ Defined augmentation pipeline  
✔ Prepared PyTorch DataLoaders  

The dataset is now ready for model training.